In [27]:
import pandas as pd

train = pd.read_csv("../house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../house-prices-advanced-regression-techniques/test.csv")

train.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Missing Values Overview


In [21]:
train.isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
                 ... 
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
SalePrice           0
Length: 81, dtype: int64

## Column Types: Categorical vs Numerical


In [22]:
from IPython.display import display

display(train.select_dtypes(include="object").columns)
display(train.select_dtypes(exclude="object").columns)



C:\Users\cicar\AppData\Local\Temp\ipykernel_3988\869671163.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(train.select_dtypes(include="object").columns)


Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold', 'SalePrice'],
      dtype='str')

# Missing value summary

In [23]:
missing_summary = train.isna().sum().sort_values(ascending=False)
missing_percentage = (train.isna().mean() * 100).sort_values(ascending=False)

pd.DataFrame({
    'missing_count': missing_summary,
    'missing_percentage': missing_percentage.round(2)
})

,missing_count,missing_percentage
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
...,...,...
MoSold,0,0.00
YrSold,0,0.00
SaleType,0,0.00
SaleCondition,0,0.00


### Drop High-Missing Columns (>80%)

The columns PoolQC (99.5%), MiscFeature (96.3%), Alley (93.8%), and Fence (80.7%)
were already removed earlier in the workflow or were not present in the loaded dataset.

These features contain extremely sparse information and provide no predictive value,
so no further action is required at this step.



In [24]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']


### Impute Remaining Missing Values

- MasVnrType: Missing values indicate no masonry veneer → imputed with "None".
- LotFrontage: Numeric and skewed → imputed with median.
- GarageType, GarageFinish, GarageQual, GarageCond: Missing values indicate no garage → imputed with "None".
- BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2: Missing values indicate no basement → imputed with "None".
- Electrical: Very few missing values → imputed with mode.


In [ ]:
# MasVnrType: missing means no masonry veneer
train['MasVnrType'] = train['MasVnrType'].fillna('None')

# LotFrontage: numeric + skewed → median
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())

# Garage-related: missing means no garage
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_cols:
    train[col] = train[col].fillna('None')

# Basement-related: missing means no basement
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in bsmt_cols:
    train[col] = train[col].fillna('None')

# Electrical: few missing → mode
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])
